# Files

> Canonical provider-agnostic Files API helpers.

## Imports

In [ ]:
#| export
# from __future__ import annotations
# from dataclasses import dataclass, field
# from pathlib import Path
# from typing import Any, Dict, List, Optional, Tuple
# import mimetypes
# from urllib.parse import urlparse
# import httpx
# from .errors import APIError, UnsupportedCapabilityError, api_error_from_http
# from .highlevel import infer_provider, mk_auto_client
# from .types import Part

In [ ]:
#| hide
from fastcore.test import *

## Purpose And Design

`files.py` offers a provider-agnostic files interface and canonical file-part construction.

### Why this module exists

File upload/download APIs vary significantly across providers. This module centralizes that divergence while keeping call sites simple.

### Architectural fit

- Upstream: model/provider selection and local file inputs.
- Downstream: provider file endpoints + canonical `Part(type="input_file")` for chat/completion flows.

### Key design choices

- Normalize file references into a single `FileRef` dataclass.
- Expose CRUD-like async operations (`afile_create/get/list/delete/content`).
- Provide `to_input_file_part(...)` to bridge uploaded files directly into message content.

### Reader outcome

After this notebook, you should know how to upload once and reuse file references across providers with minimal code changes.

### `FileRef`

Canonical file handle across providers.

**Why it exists**
- Uploaded file identifiers and metadata differ by provider.

**Design Notes**
- Unifies provider, id, filename, mime type, bytes, and raw payload in one object.

**Connections**
- Produced by `afile_*` operations.
- Converted into canonical message part via `to_input_file_part`.

In [ ]:
#| export
class FileRef:
    "Provider-agnostic file handle + metadata."
    id: str
    provider: str
    name: str = ""
    filename: str = ""
    mime_type: str = ""
    uri: str = ""
    size_bytes: Optional[int] = None
    raw: Dict[str, Any] = field(default_factory=dict)

### `_pick_first`

Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_pick_first(d: Dict[str, Any], *keys: str)`
- Returns: `Any`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _pick_first(d: Dict[str, Any], *keys: str) -> Any:
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None

### `_to_int`

Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_to_int(v: Any)`
- Returns: `Optional[int]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _to_int(v: Any) -> Optional[int]:
    try:
        if v in (None, ""):
            return None
        return int(v)
    except Exception:
        return None

### `_file_obj`

Unwrap common `{file: {...}}` envelopes.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_file_obj(raw: Dict[str, Any])`
- Returns: `Dict[str, Any]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _file_obj(raw: Dict[str, Any]) -> Dict[str, Any]:
    "Unwrap common `{file: {...}}` envelopes."
    if isinstance(raw.get("file"), dict):
        return raw["file"]
    return raw

### `_to_file_ref`

Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_to_file_ref(raw: Dict[str, Any], provider: str)`
- Returns: `FileRef`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _to_file_ref(raw: Dict[str, Any], provider: str) -> FileRef:
    obj = _file_obj(raw)
    fid = str(_pick_first(obj, "id", "file_id", "name") or "")
    nm = str(_pick_first(obj, "name", "display_name", "displayName") or "")
    fn = str(_pick_first(obj, "filename", "file_name", "display_name", "displayName") or "")
    mt = str(_pick_first(obj, "mime_type", "mimeType") or "")
    uri = str(_pick_first(obj, "uri", "file_uri", "fileUri") or "")
    sz = _to_int(_pick_first(obj, "bytes", "size_bytes", "sizeBytes"))
    return FileRef(id=fid, provider=provider, name=nm, filename=fn, mime_type=mt, uri=uri, size_bytes=sz, raw=raw)

### `_list_to_file_refs`

Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_list_to_file_refs(raw: Dict[str, Any], provider: str)`
- Returns: `List[FileRef]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _list_to_file_refs(raw: Dict[str, Any], provider: str) -> List[FileRef]:
    items = raw.get("data")
    if not isinstance(items, list):
        items = raw.get("files")
    if not isinstance(items, list):
        items = []
    return [_to_file_ref(o if isinstance(o, dict) else {"raw": o}, provider) for o in items]

### `_root_url`

Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_root_url(base_url: str)`
- Returns: `str`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _root_url(base_url: str) -> str:
    u = urlparse(base_url or "")
    if not (u.scheme and u.netloc):
        return ""
    return f"{u.scheme}://{u.netloc}"

### `_guess_mime`

Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_guess_mime(filename: str, mime_type: str)`
- Returns: `str`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _guess_mime(filename: str, mime_type: str) -> str:
    if isinstance(mime_type, str) and mime_type:
        return mime_type
    if isinstance(filename, str) and filename:
        g = mimetypes.guess_type(filename)[0]
        if isinstance(g, str) and g:
            return g
    return "application/octet-stream"

### `_read_file_bytes`

Normalize file/path/bytes-like input to `(bytes, filename, mime_type)`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_read_file_bytes(*, file: Any, filename: str, mime_type: str)`
- Returns: `Tuple[bytes, str, str]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
def _read_file_bytes(*, file: Any = None, filename: str = "", mime_type: str = "") -> Tuple[bytes, str, str]:
    "Normalize file/path/bytes-like input to `(bytes, filename, mime_type)`."
    data: bytes
    fn = filename or ""
    if file is None:
        raise ValueError("`file` is required for binary upload. Pass bytes, path, or a binary file object.")

    if isinstance(file, (str, Path)):
        p = Path(file)
        data = p.read_bytes()
        if not fn:
            fn = p.name
    elif isinstance(file, (bytes, bytearray)):
        data = bytes(file)
    elif hasattr(file, "read"):
        data = file.read()
        if not isinstance(data, (bytes, bytearray)):
            raise TypeError("File-like object `.read()` must return bytes.")
        data = bytes(data)
        if not fn:
            fn = getattr(file, "name", "") or ""
            if isinstance(fn, str) and fn and "/" in fn:
                fn = Path(fn).name
    else:
        raise TypeError("Unsupported `file` type. Use bytes, path-like, or binary file object.")

    if not fn:
        fn = "upload.bin"
    mt = _guess_mime(fn, mime_type)
    return data, fn, mt

### `to_input_file_part`

Converts a `FileRef` into canonical `Part(type="input_file")` payload.

**Why it exists**
- Chat/completion paths should consume one canonical file part shape regardless of upload backend.

**Design Notes**
- Embeds provider-appropriate metadata so serializers can map correctly downstream.

**Connections**
- Used after `afile_create` in document/image/audio/video workflows.

In [ ]:
#| export
def to_input_file_part(f: FileRef | Dict[str, Any] | str, *, provider: str = "") -> Part:
    "Create a canonical `Part(type='input_file', ...)` from a file reference."
    if isinstance(f, FileRef):
        ref = f
    elif isinstance(f, dict):
        p = provider or str(f.get("provider") or "")
        ref = FileRef(
            id=str(f.get("id") or f.get("name") or ""),
            provider=p,
            name=str(f.get("name") or ""),
            filename=str(f.get("filename") or f.get("display_name") or ""),
            mime_type=str(f.get("mime_type") or f.get("mimeType") or ""),
            uri=str(f.get("uri") or f.get("file_uri") or f.get("fileUri") or ""),
            raw=dict(f),
        )
    else:
        ref = FileRef(id=str(f), provider=provider or "")

    prov = (provider or ref.provider or "").strip().lower()
    if prov == "anthropic":
        if not ref.id:
            raise ValueError("Anthropic file part requires `FileRef.id`.")
        return Part(
            type="input_file",
            data={
                "file_id": ref.id,
                "anthropic": {"type": "document", "source": {"type": "file", "file_id": ref.id}},
            },
        )
    if prov == "gemini":
        uri = ref.uri or ref.id
        if not uri:
            raise ValueError("Gemini file part requires `FileRef.uri` or `FileRef.id`.")
        mt = ref.mime_type or "application/octet-stream"
        return Part(
            type="input_file",
            data={
                "file_url": uri,
                "mimeType": mt,
                "gemini": {"fileData": {"fileUri": uri, "mimeType": mt}},
            },
        )
    # OpenAI / openai_chat / openai_compat default.
    d: Dict[str, Any] = {}
    if ref.id:
        d["file_id"] = ref.id
    elif ref.uri:
        d["file_url"] = ref.uri
    if ref.filename:
        d["filename"] = ref.filename
    if ref.mime_type:
        d["mimeType"] = ref.mime_type
    return Part(type="input_file", data=d or None)

### `_openai_create`

Async Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_openai_create(c, *, file: Any, filename: str, mime_type: str, purpose: str, headers: Optional[Dict[str, str]])`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def _openai_create(c, *, file: Any, filename: str, mime_type: str, purpose: str, headers: Optional[Dict[str, str]] = None):
    b, fn, mt = _read_file_bytes(file=file, filename=filename, mime_type=mime_type)
    return await c.api.call("/files", "POST", headers=headers, body=None, data={"purpose": purpose}, files={"file": (fn, b, mt)})

### `_anthropic_create`

Async Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_anthropic_create(c, *, file: Any, filename: str, mime_type: str, purpose: str, headers: Optional[Dict[str, str]])`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: primarily local logic.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def _anthropic_create(c, *, file: Any, filename: str, mime_type: str, purpose: str, headers: Optional[Dict[str, str]] = None):
    b, fn, mt = _read_file_bytes(file=file, filename=filename, mime_type=mime_type)
    h = dict(headers or {})
    h.setdefault("anthropic-beta", "files-api-2025-04-14")
    return await c.api.call("/v1/files", "POST", headers=h, body=None, data={"purpose": purpose}, files={"file": (fn, b, mt)})

### `_gemini_upload_resumable`

Async Function in `files.py`.

**Role In This Module**
- Internal helper for module composition.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `_gemini_upload_resumable(c, *, file: Any, filename: str, mime_type: str, headers: Optional[Dict[str, str]])`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: `errors`.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def _gemini_upload_resumable(c, *, file: Any, filename: str, mime_type: str, headers: Optional[Dict[str, str]] = None):
    b, fn, mt = _read_file_bytes(file=file, filename=filename, mime_type=mime_type)
    root = _root_url(c.api.base_url)
    if not root:
        raise ValueError("Could not infer Gemini API root URL for resumable upload.")
    start_url = f"{root}/upload/v1beta/files"

    h1 = {
        "X-Goog-Upload-Protocol": "resumable",
        "X-Goog-Upload-Command": "start",
        "X-Goog-Upload-Header-Content-Length": str(len(b)),
        "X-Goog-Upload-Header-Content-Type": mt,
        "Content-Type": "application/json",
        **(headers or {}),
    }
    # Start resumable session.
    r = await c.api.transport.request("POST", start_url, headers=h1, json_data={"file": {"display_name": fn}}, raw=True)
    upload_url = r.headers.get("x-goog-upload-url")
    if not upload_url:
        raise UnsupportedCapabilityError("Gemini upload session did not return `x-goog-upload-url`.")

    h2 = {
        "Content-Length": str(len(b)),
        "X-Goog-Upload-Offset": "0",
        "X-Goog-Upload-Command": "upload, finalize",
        **(headers or {}),
    }
    return await c.api.transport.request("POST", upload_url, headers=h2, data=b)

### `afile_create`

Provider-agnostic async file upload entrypoint.

**Why it exists**
- Upload APIs vary widely (multipart, resumable, purpose fields, etc.).

**Design Notes**
- Dispatches by inferred provider family and normalizes outputs to `FileRef`.

**Connections**
- Starting point for canonical file workflows in `fastllm`.

In [ ]:
#| export
async def afile_create(
    model: str,
    *,
    file: Any = None,
    filename: str = "",
    mime_type: str = "",
    purpose: str = "user_data",
    provider: str = "",
    api_key: str = "",
    base_url: str = "",
    timeout: float = 60.0,
    headers: Optional[Dict[str, str]] = None,
) -> FileRef:
    "Create/upload a file and return canonical `FileRef`."
    c = mk_auto_client(model=model, provider=provider, api_key=api_key, base_url=base_url, timeout=timeout)
    fam = infer_provider(model, provider=provider, base_url=base_url)
    try:
        if fam in ("openai", "openai_compat"):
            raw = await _openai_create(c, file=file, filename=filename, mime_type=mime_type, purpose=purpose, headers=headers)
            return _to_file_ref(raw if isinstance(raw, dict) else {"raw": raw}, c.provider or fam)
        if fam == "anthropic":
            raw = await _anthropic_create(c, file=file, filename=filename, mime_type=mime_type, purpose=purpose, headers=headers)
            return _to_file_ref(raw if isinstance(raw, dict) else {"raw": raw}, "anthropic")
        if fam == "gemini":
            raw = await _gemini_upload_resumable(c, file=file, filename=filename, mime_type=mime_type, headers=headers)
            return _to_file_ref(raw if isinstance(raw, dict) else {"raw": raw}, "gemini")
        raise UnsupportedCapabilityError(f"Files API not supported for inferred provider: {fam}")
    except APIError:
        raise
    except httpx.HTTPStatusError as e:
        raise api_error_from_http(e, provider=c.provider or fam, model=model, endpoint="files.create")
    finally:
        await c.aclose()

### `afile_get`

Get file metadata as canonical `FileRef`.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `afile_get(model: str, file_id: str, *, provider: str, api_key: str, base_url: str, timeout: float, headers: Optional[Dict[str, str]])`
- Returns: `FileRef`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: `errors`, `highlevel`.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def afile_get(
    model: str,
    file_id: str,
    *,
    provider: str = "",
    api_key: str = "",
    base_url: str = "",
    timeout: float = 60.0,
    headers: Optional[Dict[str, str]] = None,
) -> FileRef:
    "Get file metadata as canonical `FileRef`."
    c = mk_auto_client(model=model, provider=provider, api_key=api_key, base_url=base_url, timeout=timeout)
    fam = infer_provider(model, provider=provider, base_url=base_url)
    try:
        if fam in ("openai", "openai_compat"):
            raw = await c.api.call("/files/{file_id}", "GET", headers=headers, route={"file_id": file_id})
            return _to_file_ref(raw if isinstance(raw, dict) else {"raw": raw}, c.provider or fam)
        if fam == "anthropic":
            h = dict(headers or {})
            h.setdefault("anthropic-beta", "files-api-2025-04-14")
            raw = await c.api.call("/v1/files/{file_id}", "GET", headers=h, route={"file_id": file_id})
            return _to_file_ref(raw if isinstance(raw, dict) else {"raw": raw}, "anthropic")
        if fam == "gemini":
            nm = file_id if str(file_id).startswith("files/") else f"files/{file_id}"
            raw = await c.api.files.get(name=nm, _headers=headers)
            return _to_file_ref(raw if isinstance(raw, dict) else {"raw": raw}, "gemini")
        raise UnsupportedCapabilityError(f"Files API not supported for inferred provider: {fam}")
    except APIError:
        raise
    except httpx.HTTPStatusError as e:
        raise api_error_from_http(e, provider=c.provider or fam, model=model, endpoint="files.get")
    finally:
        await c.aclose()

### `afile_list`

List files as canonical `FileRef` objects.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `afile_list(model: str, *, provider: str, api_key: str, base_url: str, timeout: float, headers: Optional[Dict[str, str]], **query)`
- Returns: `List[FileRef]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: `errors`, `highlevel`.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def afile_list(
    model: str,
    *,
    provider: str = "",
    api_key: str = "",
    base_url: str = "",
    timeout: float = 60.0,
    headers: Optional[Dict[str, str]] = None,
    **query,
) -> List[FileRef]:
    "List files as canonical `FileRef` objects."
    c = mk_auto_client(model=model, provider=provider, api_key=api_key, base_url=base_url, timeout=timeout)
    fam = infer_provider(model, provider=provider, base_url=base_url)
    try:
        if fam in ("openai", "openai_compat"):
            raw = await c.api.call("/files", "GET", headers=headers, query=query or None)
            return _list_to_file_refs(raw if isinstance(raw, dict) else {"data": []}, c.provider or fam)
        if fam == "anthropic":
            h = dict(headers or {})
            h.setdefault("anthropic-beta", "files-api-2025-04-14")
            raw = await c.api.call("/v1/files", "GET", headers=h, query=query or None)
            return _list_to_file_refs(raw if isinstance(raw, dict) else {"data": []}, "anthropic")
        if fam == "gemini":
            raw = await c.api.files.list(_headers=headers, **query)
            return _list_to_file_refs(raw if isinstance(raw, dict) else {"files": []}, "gemini")
        raise UnsupportedCapabilityError(f"Files API not supported for inferred provider: {fam}")
    except APIError:
        raise
    except httpx.HTTPStatusError as e:
        raise api_error_from_http(e, provider=c.provider or fam, model=model, endpoint="files.list")
    finally:
        await c.aclose()

### `afile_delete`

Delete a file and return provider raw response.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `afile_delete(model: str, file_id: str, *, provider: str, api_key: str, base_url: str, timeout: float, headers: Optional[Dict[str, str]])`
- Returns: `Dict[str, Any]`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: `errors`, `highlevel`.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def afile_delete(
    model: str,
    file_id: str,
    *,
    provider: str = "",
    api_key: str = "",
    base_url: str = "",
    timeout: float = 60.0,
    headers: Optional[Dict[str, str]] = None,
) -> Dict[str, Any]:
    "Delete a file and return provider raw response."
    c = mk_auto_client(model=model, provider=provider, api_key=api_key, base_url=base_url, timeout=timeout)
    fam = infer_provider(model, provider=provider, base_url=base_url)
    try:
        if fam in ("openai", "openai_compat"):
            raw = await c.api.call("/files/{file_id}", "DELETE", headers=headers, route={"file_id": file_id})
            return raw if isinstance(raw, dict) else {"raw": raw}
        if fam == "anthropic":
            h = dict(headers or {})
            h.setdefault("anthropic-beta", "files-api-2025-04-14")
            raw = await c.api.call("/v1/files/{file_id}", "DELETE", headers=h, route={"file_id": file_id})
            return raw if isinstance(raw, dict) else {"raw": raw}
        if fam == "gemini":
            nm = file_id if str(file_id).startswith("files/") else f"files/{file_id}"
            raw = await c.api.files.delete(name=nm, _headers=headers)
            return raw if isinstance(raw, dict) else {"raw": raw}
        raise UnsupportedCapabilityError(f"Files API not supported for inferred provider: {fam}")
    except APIError:
        raise
    except httpx.HTTPStatusError as e:
        raise api_error_from_http(e, provider=c.provider or fam, model=model, endpoint="files.delete")
    finally:
        await c.aclose()

### `afile_content`

Download file content bytes when the provider exposes a file-content endpoint.

**Role In This Module**
- Public symbol in this module API surface.
- Carries behavior central to this module stage of the pipeline.

**Design Notes**
- Signature: `afile_content(model: str, file_id: str, *, provider: str, api_key: str, base_url: str, timeout: float, headers: Optional[Dict[str, str]])`
- Returns: `bytes`
- Small helper/function boundary keeps calling layers easier to compose and test.

**Connections**
- In-module dependencies used: `errors`, `highlevel`.
- Referenced by modules: none or only via orchestration.
- Module upstream: `errors`, `highlevel`, `types`.

In [ ]:
#| export
async def afile_content(
    model: str,
    file_id: str,
    *,
    provider: str = "",
    api_key: str = "",
    base_url: str = "",
    timeout: float = 60.0,
    headers: Optional[Dict[str, str]] = None,
) -> bytes:
    "Download file content bytes when the provider exposes a file-content endpoint."
    c = mk_auto_client(model=model, provider=provider, api_key=api_key, base_url=base_url, timeout=timeout)
    fam = infer_provider(model, provider=provider, base_url=base_url)
    try:
        if fam in ("openai", "openai_compat"):
            resp = await c.api.call("/files/{file_id}/content", "GET", headers=headers, route={"file_id": file_id}, raw=True)
            if isinstance(resp, httpx.Response):
                return resp.content
            if isinstance(resp, (bytes, bytearray)):
                return bytes(resp)
            return str(resp).encode()
        if fam == "anthropic":
            h = dict(headers or {})
            h.setdefault("anthropic-beta", "files-api-2025-04-14")
            resp = await c.api.call("/v1/files/{file_id}/content", "GET", headers=h, route={"file_id": file_id}, raw=True)
            if isinstance(resp, httpx.Response):
                return resp.content
            if isinstance(resp, (bytes, bytearray)):
                return bytes(resp)
            return str(resp).encode()
        if fam == "gemini":
            raise UnsupportedCapabilityError("Gemini file content download is not exposed in the current built-in ops.")
        raise UnsupportedCapabilityError(f"Files API not supported for inferred provider: {fam}")
    except APIError:
        raise
    except httpx.HTTPStatusError as e:
        raise api_error_from_http(e, provider=c.provider or fam, model=model, endpoint="files.content")
    finally:
        await c.aclose()

### Export Surface

In [ ]:
#| export
__all__ = "FileRef to_input_file_part afile_create afile_get afile_list afile_delete afile_content".split()

## Quick Checks

In [ ]:
import fastllm.files as m
for nm in ['FileRef', 'to_input_file_part', 'afile_create', 'afile_get', 'afile_list', 'afile_delete', 'afile_content']: assert hasattr(m, nm), nm
from fastllm.files import FileRef, to_input_file_part
f = FileRef(provider='openai', id='file_1', filename='a.txt', mime_type='text/plain')
p = to_input_file_part(f)
assert p.type=='input_file'